# Statistical Classification Log-Loss (Cross-Entropy)  
### Use case: Hours Studied vs. Pass / Fail  

**Workshop:** Logistic Regression Classifier Workshop  
**Team A (us):**  

| Member | Student ID |
|---|---|
| Sultan | 9114837 |
| Antonio | 9072844 |
| John | 9115726 |
| Eche | 9078881 |

**Our repo:** _add link_  
**Team B (peer group):** _add names_  
**Team B's repo:** _add link_  

---  

**Goal of this notebook:** understand and implement **Log-Loss**, the score we use to judge how good a classifier's *probabilities* are.  

| Step | What we do |
|---|---|
| 1 | Look at the data |
| 2 | Meet the sigmoid function (turns any number into a probability) |
| 3 | Train a logistic regression model to get probabilities |
| 4 | Understand Log-Loss for **one** student |
| 5 | Implement Log-Loss for the **whole** dataset (from scratch) |
| 6 | See why Log-Loss tells us more than accuracy |
| 7 | See how a model *uses* Log-Loss to learn (gradient descent) |
| 8 | Use the model on new students |
| 9 | Talking points + who presents what |
| 10 | Notes from the session with Team B |
| 11 | Report details for submission |

## Setup  
- **NumPy**: math on arrays of numbers  
- **pandas**: tables (DataFrames)  
- **Plotly**: interactive charts (hover over points to see values)  
- **scikit-learn**: ready-made logistic regression and a built-in `log_loss` we use to check our own version  

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, log_loss as sklearn_log_loss

FAIL_COLOR = "#E4572E"
PASS_COLOR = "#2E86AB"
CURVE_COLOR = "#6A4C93"

## 1. The data  

Each row is one student.  
- **`hours`** = how many hours they studied (the **feature**, `x`)  
- **`passed`** = the real outcome (the **label**, `y`): `1` = Pass, `0` = Fail  

This is a **binary classification** problem: the answer is one of two classes.  

In [ ]:
hours = np.array([0.5, 0.75, 1, 1.25, 1.5, 1.75, 2, 2.25, 2.5, 2.75,
                  3, 3.25, 3.5, 3.75, 4, 4.25, 4.5, 4.75, 5])
passed = np.array([0, 0, 0, 0, 0, 0, 0, 0, 1, 0,
                   1, 0, 1, 1, 1, 1, 1, 1, 1])

df = pd.DataFrame({"hours": hours, "passed": passed})
print(f"Students: {len(df)}  |  Passed: {passed.sum()}  |  Failed: {(passed == 0).sum()}")
df.T

**Reading the chart below**  
- **x-axis**: hours studied  
- **y-axis**: outcome (0 = Fail, 1 = Pass)  
- **Red dots** = students who failed, **blue dots** = students who passed  
- Notice the **overlap zone (≈ 2.5 to 3.25 hours)**: some students passed and some failed with similar hours. No straight cut-off separates them perfectly, so the model has to work with *probabilities*.  

In [ ]:
fig = go.Figure()
for label, name, color in [(0, "Fail", FAIL_COLOR), (1, "Pass", PASS_COLOR)]:
    sub = df[df.passed == label]
    fig.add_trace(go.Scatter(x=sub.hours, y=sub.passed, mode="markers", name=name,
                             marker=dict(size=13, color=color, line=dict(width=1, color="white"))))
fig.add_vrect(x0=2.4, x1=3.35, fillcolor="gray", opacity=0.12, line_width=0,
              annotation_text="overlap zone", annotation_position="top left")
fig.update_layout(title="Hours Studied vs. Exam Outcome", template="plotly_white",
                  xaxis_title="Hours studied", yaxis_title="Outcome (0 = Fail, 1 = Pass)",
                  yaxis=dict(tickvals=[0, 1]), height=420)
fig.show()

## 2. The sigmoid function  

Logistic regression first draws a straight line:  

$$ z = w_0 + w_1 \cdot x $$

- $w_0$ = **intercept** (bias), $w_1$ = **weight** for hours studied, $x$ = hours studied  
- $z$ can be any number (−100, 0, 7.3 ...), so it is not a probability yet.  

The **sigmoid** squeezes $z$ into the range 0 to 1, so we can read it as a probability:  

$$ p = \sigma(z) = \frac{1}{1 + e^{-z}} $$

| z | sigmoid(z) | meaning |
|---|---|---|
| very negative | close to 0 | very likely **Fail** |
| 0 | 0.5 | unsure (50/50) |
| very positive | close to 1 | very likely **Pass** |

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

for z in [-5, -1, 0, 1, 5]:
    print(f"sigmoid({z:>2}) = {sigmoid(z):.3f}")

**Reading the chart below**  
- **Purple S-curve**: the sigmoid output for every value of `z`  
- **Dashed line at 0.5**: the **decision threshold**. Above it → predict Pass, below it → predict Fail  
- The curve never touches 0 or 1, it only gets very close  

In [ ]:
z = np.linspace(-8, 8, 200)
fig = go.Figure(go.Scatter(x=z, y=sigmoid(z), mode="lines", name="sigmoid(z)",
                           line=dict(color=CURVE_COLOR, width=4)))
fig.add_hline(y=0.5, line_dash="dash", line_color="gray", annotation_text="threshold = 0.5")
fig.add_annotation(x=-5, y=0.15, text="predict Fail", showarrow=False, font=dict(color=FAIL_COLOR, size=14))
fig.add_annotation(x=5, y=0.85, text="predict Pass", showarrow=False, font=dict(color=PASS_COLOR, size=14))
fig.update_layout(title="The Sigmoid Function", template="plotly_white",
                  xaxis_title="z  (output of the straight line)", yaxis_title="Probability", height=420)
fig.show()

## 3. Train a logistic regression model  

`scikit-learn` finds the best $w_0$ and $w_1$ for us.  
`reshape(-1, 1)` turns `hours` into a single column, the shape scikit-learn expects for features.  

The **decision boundary** is the number of hours where the probability is exactly 0.5 (where $z = 0$):  

$$ x_{boundary} = -\frac{w_0}{w_1} $$

In [ ]:
X = hours.reshape(-1, 1)
y = passed

model = LogisticRegression()
model.fit(X, y)

w0 = model.intercept_[0]
w1 = model.coef_[0][0]
boundary = -w0 / w1

print(f"Intercept w0 = {w0:.3f}")
print(f"Weight    w1 = {w1:.3f}")
print(f"Decision boundary = {boundary:.2f} hours")

Now we ask the model for a **probability of passing** for every student.  

- `predict_proba(X)` returns two columns: `[P(Fail), P(Pass)]`. We keep column `1` = **P(Pass)**  
- `predict(X)` applies the 0.5 threshold and returns 0 or 1  

In [ ]:
p_pass = model.predict_proba(X)[:, 1]
predicted = model.predict(X)

results = pd.DataFrame({
    "hours": hours,
    "actual": y,
    "P(pass)": p_pass.round(3),
    "predicted": predicted,
    "correct?": np.where(predicted == y, "yes", "NO"),
})
print(f"Accuracy: {accuracy_score(y, predicted):.2%}")
results

**Reading the chart below**  
- **Dots**: real students (red = failed, blue = passed)  
- **Purple S-curve**: the model's probability of passing for any number of hours  
- **Green dashed vertical line**: decision boundary. Right of it → predicted Pass  
- **Gray dashed horizontal line**: the 0.5 threshold  
- **Circled dots**: students the model got **wrong**. These will create the biggest Log-Loss later.  

In [ ]:
x_line = np.linspace(0, 6, 300).reshape(-1, 1)
p_line = model.predict_proba(x_line)[:, 1]
wrong = results[results["correct?"] == "NO"]

fig = go.Figure()
fig.add_trace(go.Scatter(x=x_line.ravel(), y=p_line, mode="lines", name="P(pass)",
                         line=dict(color=CURVE_COLOR, width=4)))
for label, name, color in [(0, "Fail", FAIL_COLOR), (1, "Pass", PASS_COLOR)]:
    sub = results[results.actual == label]
    fig.add_trace(go.Scatter(x=sub.hours, y=sub.actual, mode="markers", name=name,
                             marker=dict(size=12, color=color)))
fig.add_trace(go.Scatter(x=wrong.hours, y=wrong.actual, mode="markers", name="Misclassified",
                         marker=dict(size=24, color="rgba(0,0,0,0)", line=dict(width=2.5, color="black"))))
fig.add_vline(x=boundary, line_dash="dash", line_color="green",
              annotation_text=f"boundary = {boundary:.2f} h")
fig.add_hline(y=0.5, line_dash="dot", line_color="gray")
fig.update_layout(title="Logistic Regression: Probability of Passing", template="plotly_white",
                  xaxis_title="Hours studied", yaxis_title="Probability of passing", height=450)
fig.show()

## 4. Log-Loss for ONE student  

Accuracy only asks *"right or wrong?"*.  
**Log-Loss asks *"how confident were you, and were you right?"***  

Think of it as a **surprise score**: how surprised the model is when it sees the real answer.  

**Two cases** ($p$ = predicted probability of passing):  

$$
\text{loss} =
\begin{cases}
-\log(p) & \text{if the student passed } (y = 1) \\
-\log(1 - p) & \text{if the student failed } (y = 0)
\end{cases}
$$

**Same thing in one line** (when $y=1$ the right part disappears, when $y=0$ the left part disappears):  

$$ \text{loss} = -\big[\, y \cdot \log(p) + (1 - y)\cdot \log(1 - p) \,\big] $$

`log` here is the natural log (`np.log`).  

**Why "clip"?** `log(0)` is minus infinity and would crash the math.  
`np.clip` keeps `p` slightly inside the range, between `0.000000000000001` and `0.999999999999999`.  

In [ ]:
EPS = 1e-15

def single_log_loss(y_true, p):
    p = np.clip(p, EPS, 1 - EPS)
    return -(y_true * np.log(p) + (1 - y_true) * np.log(1 - p))

Let's test four situations: confident & right, confident & wrong, for each outcome.  

In [ ]:
cases = pd.DataFrame({
    "situation": ["Passed, model said 95% pass", "Passed, model said 10% pass",
                  "Failed, model said 5% pass",  "Failed, model said 90% pass"],
    "y": [1, 1, 0, 0],
    "p": [0.95, 0.10, 0.05, 0.90],
})
cases["loss"] = single_log_loss(cases.y, cases.p).round(3)
cases["verdict"] = ["good (tiny loss)", "bad (big loss)", "good (tiny loss)", "bad (big loss)"]
cases

**Takeaway:** a confident correct answer costs about **0.05**. A confident wrong answer costs about **2.3**, roughly **45× more**.  

**Reading the chart below**  
- **Left panel (green)**: the student really **passed**. As predicted P(pass) moves toward 1, loss drops to 0. As it moves toward 0, loss shoots up.  
- **Right panel (orange)**: the student really **failed**. It is the mirror image: loss is low near 0 and explodes near 1.  
- The steep wall at one end is the key idea: **being confidently wrong is punished very hard**.  

In [ ]:
p = np.linspace(0.001, 0.999, 300)
fig = make_subplots(rows=1, cols=2, subplot_titles=("Student PASSED  (y = 1)", "Student FAILED  (y = 0)"))
fig.add_trace(go.Scatter(x=p, y=-np.log(p), mode="lines", name="-log(p)",
                         line=dict(color="#2CA58D", width=4)), row=1, col=1)
fig.add_trace(go.Scatter(x=p, y=-np.log(1 - p), mode="lines", name="-log(1-p)",
                         line=dict(color="#F18F01", width=4)), row=1, col=2)
fig.add_annotation(x=0.15, y=4, text="confident & WRONG<br>→ huge loss", showarrow=False, row=1, col=1)
fig.add_annotation(x=0.8, y=0.7, text="confident & right<br>→ tiny loss", showarrow=False, row=1, col=1)
fig.add_annotation(x=0.85, y=4, text="confident & WRONG<br>→ huge loss", showarrow=False, row=1, col=2)
fig.add_annotation(x=0.2, y=0.7, text="confident & right<br>→ tiny loss", showarrow=False, row=1, col=2)
fig.update_xaxes(title_text="Predicted probability of passing (p)")
fig.update_yaxes(title_text="Log-Loss", col=1)
fig.update_layout(title="Log-Loss Penalty Curves", template="plotly_white", height=430, showlegend=False)
fig.show()

## 5. Log-Loss for the WHOLE dataset (the algorithm)  

We average the single-student loss over all $N$ students:  

$$ \text{LogLoss} = -\frac{1}{N} \sum_{i=1}^{N} \big[\, y_i \log(p_i) + (1 - y_i)\log(1 - p_i) \,\big] $$

**Algorithm**  
1. **Input**: real labels `y_true` (0/1) and predicted probabilities `y_prob` (P of class 1)  
2. **Clip** probabilities away from exactly 0 and 1  
3. **Compute** the loss for each student with the one-line formula  
4. **Average** all the losses  
5. **Return** one number. **Lower is better** (0 = perfect)  

In [ ]:
def log_loss(y_true, y_prob):
    y_true = np.asarray(y_true)
    y_prob = np.clip(np.asarray(y_prob), EPS, 1 - EPS)
    losses = -(y_true * np.log(y_prob) + (1 - y_true) * np.log(1 - y_prob))
    return losses.mean()

**Check:** our function should give the same answer as scikit-learn's built-in `log_loss`.  

In [ ]:
ours = log_loss(y, p_pass)
theirs = sklearn_log_loss(y, p_pass)

print(f"Our Log-Loss:          {ours:.4f}")
print(f"scikit-learn Log-Loss: {theirs:.4f}")
print("Match!" if np.isclose(ours, theirs) else "Mismatch")

**What does 0.275 mean?** A useful reference: a model that always says **50%** (pure guessing) scores $-\log(0.5) \approx 0.693$.  
Our model is well below that, so its probabilities are much better than guessing.  

Next, let's see **which students** contribute the most loss.  

In [ ]:
results["loss"] = single_log_loss(y, p_pass).round(3)
results.sort_values("loss", ascending=False).head(6)

**Reading the chart below**  
- Each **bar** = one student's Log-Loss  
- **Bar color** = real outcome (red = failed, blue = passed)  
- **Tallest bars (≈ 1.0)**: the two misclassified students, 2.5 h (passed) and 3.25 h (failed)  
- **Next tallest (≈ 0.6)**: 2.75 h and 3.0 h, correct but very close to the boundary, so the model was unsure  
- **Dashed line** = the average, i.e. the final Log-Loss score  
- Students far from the boundary (very few or very many hours) have almost zero loss because the model was confident **and** right  

In [ ]:
colors = [PASS_COLOR if v == 1 else FAIL_COLOR for v in results.actual]
fig = go.Figure(go.Bar(x=results.hours.astype(str), y=results.loss, marker_color=colors,
                       customdata=results["P(pass)"],
                       hovertemplate="hours=%{x}<br>loss=%{y}<br>P(pass)=%{customdata}<extra></extra>"))
fig.add_hline(y=ours, line_dash="dash", line_color="black", annotation_text=f"average = {ours:.3f}")
fig.update_layout(title="Log-Loss per Student", template="plotly_white",
                  xaxis_title="Hours studied (one bar per student)", yaxis_title="Log-Loss", height=430)
fig.show()

## 6. Why not just use accuracy?  

We compare three "models" on the same students:  

| Model | How it predicts |
|---|---|
| **Coin flip** | always says 50% pass |
| **Our model** | the trained logistic regression |
| **Over-confident** | same boundary as ours, but its S-curve is 6× steeper, so its probabilities are pushed to almost 0 or 1 |

The over-confident model makes **exactly the same Pass/Fail decisions** as our model, so the **accuracy is identical**.  
(The coin flip says exactly 0.5 for everyone, which counts as "Pass", so it only gets the 9 students who passed right.)  
Only Log-Loss can tell them apart.  

In [ ]:
p_coin = np.full_like(p_pass, 0.5)
p_over = sigmoid(6 * (w0 + w1 * hours))

compare = pd.DataFrame({
    "model": ["Coin flip", "Our model", "Over-confident"],
    "accuracy": [accuracy_score(y, (p >= 0.5).astype(int)) for p in [p_coin, p_pass, p_over]],
    "log_loss": [log_loss(y, p) for p in [p_coin, p_pass, p_over]],
}).round(3)
compare

**Reading the chart below**  
- **Left**: accuracy. Our model and the over-confident model look **equal**.  
- **Right**: Log-Loss. The over-confident model is **worse**, because its few mistakes were made with near-100% confidence.  
- **Lesson:** Log-Loss rewards a model that is *honest about its uncertainty*. That matters when the probability itself is used, e.g. "this student has a 40% chance of passing, let's offer tutoring".  

In [ ]:
bar_colors = ["#9E9E9E", CURVE_COLOR, "#D1495B"]
fig = make_subplots(rows=1, cols=2, subplot_titles=("Accuracy (higher is better)", "Log-Loss (lower is better)"))
fig.add_trace(go.Bar(x=compare.model, y=compare.accuracy, marker_color=bar_colors,
                     text=compare.accuracy, textposition="outside"), row=1, col=1)
fig.add_trace(go.Bar(x=compare.model, y=compare.log_loss, marker_color=bar_colors,
                     text=compare.log_loss, textposition="outside"), row=1, col=2)
fig.update_layout(title="Same Accuracy, Different Log-Loss", template="plotly_white",
                  height=430, showlegend=False)
fig.show()

## 7. How a model learns by minimizing Log-Loss  

Training a logistic regression **means**: find the $w_0$ and $w_1$ that give the **lowest Log-Loss**.  
scikit-learn did that for us. Here we do it by hand with **gradient descent**:  

1. Start with $w_0 = 0$, $w_1 = 0$ (every student gets 50%)  
2. Compute probabilities $p$ with the sigmoid  
3. Compute the **gradients** (which direction makes the loss go down):  
   $\quad \dfrac{\partial L}{\partial w_0} = \text{mean}(p - y) \qquad \dfrac{\partial L}{\partial w_1} = \text{mean}\big((p - y)\cdot x\big)$  
4. Take a small step downhill: $w \leftarrow w - \text{learning rate} \times \text{gradient}$  
5. Repeat many times  

`p - y` is simply the **error** for each student, which is why this update is so short.  

In [ ]:
def train_with_gradient_descent(x, y, learning_rate=0.1, steps=20000):
    b0, b1 = 0.0, 0.0
    history = []
    for _ in range(steps):
        p = sigmoid(b0 + b1 * x)
        error = p - y
        b0 -= learning_rate * error.mean()
        b1 -= learning_rate * (error * x).mean()
        history.append(log_loss(y, p))
    return b0, b1, history

gd_w0, gd_w1, history = train_with_gradient_descent(hours, y)

print(f"Gradient descent: w0 = {gd_w0:.3f}, w1 = {gd_w1:.3f}, Log-Loss = {history[-1]:.4f}")
print(f"scikit-learn:     w0 = {w0:.3f}, w1 = {w1:.3f}, Log-Loss = {ours:.4f}")

**Why are the numbers different?** By default scikit-learn adds a **penalty (regularization)** that keeps the weights from growing too large, which helps the model generalize to new students. Our hand-made version has no penalty, so its weights come out about twice as large and its Log-Loss *on this training data* is lower (≈ 0.232 vs ≈ 0.275). Lower training loss is not automatically better: the penalty trades a little training loss for a model that is less over-confident.  

**Reading the chart below**  
- **x-axis**: training step number  
- **y-axis**: Log-Loss at that step  
- It **starts at ≈ 0.693** (the coin-flip value, because both weights start at 0)  
- It **drops fast** at first, then **flattens** as the model reaches the best weights it can find  

In [ ]:
fig = go.Figure(go.Scatter(y=history, mode="lines", line=dict(color=CURVE_COLOR, width=3)))
fig.add_hline(y=np.log(2), line_dash="dot", line_color="gray", annotation_text="coin flip (0.693)")
fig.update_layout(title="Gradient Descent: Log-Loss Going Down", template="plotly_white",
                  xaxis_title="Training step", yaxis_title="Log-Loss", height=420)
fig.show()

## 8. Using the model on new students  

For each new student we show:  
- **P(pass)** from the model and the **predicted class** (0.5 threshold)  
- **Loss if they pass** and **loss if they fail**: how "surprised" the model would be by each real result  

In [ ]:
new_hours = np.array([1.0, 2.6, 3.0, 4.0])
new_p = model.predict_proba(new_hours.reshape(-1, 1))[:, 1]

pd.DataFrame({
    "hours": new_hours,
    "P(pass)": new_p.round(3),
    "prediction": np.where(new_p >= 0.5, "Pass", "Fail"),
    "loss if passes": single_log_loss(1, new_p).round(3),
    "loss if fails": single_log_loss(0, new_p).round(3),
})

Near the boundary (2.6 h, 3 h) the two possible losses are **close to each other**: the model is unsure, so neither outcome surprises it much.  
Far from the boundary (1 h, 4 h) one loss is tiny and the other is large: the model has committed to an answer.  

## 9. Talking Points (Team A)  

**1. Log-Loss measures confidence, not just correctness.**  
Accuracy treats every mistake the same. Log-Loss uses $-\log(p)$, so a confident wrong prediction (e.g. 10% pass for a student who passed → loss 2.3) costs about 45× more than a confident right one (≈ 0.05). Section 6 shows two models with **identical accuracy** but different Log-Loss.  

**2. The implementation is three steps: clip, compute, average.**  
`log_loss()` clips probabilities to avoid `log(0)`, applies $-[y\log p + (1-y)\log(1-p)]$ to every student, and returns the mean. Our result (**≈ 0.275**) matches scikit-learn exactly, and is well below the coin-flip baseline of **0.693**. The per-student chart shows most of the loss comes from the 4 students in the 2.5 to 3.25 h overlap zone.  

**3. Log-Loss is what the model minimizes during training.**  
Gradient descent starts at 0.693 and repeatedly nudges $w_0, w_1$ in the direction of `mean(p - y)` until the loss stops dropping. It reaches ≈ 0.232; scikit-learn stops at ≈ 0.275 because its default regularization penalty keeps the weights smaller on purpose.  

### Who presents what  

| Part of the walkthrough | Notebook sections | Presenter |
|---|---|---|
| Data, sigmoid and the trained model | 1, 2, 3 | |
| Talking point 1: confidence vs correctness | 4, 6 | |
| Talking point 2: the `log_loss()` implementation | 5 | |
| Talking point 3: gradient descent + new predictions | 7, 8 | |

## 10. Session Notes with Team B  

### a) Our review of Team B's code (step 4a)  
**Questions we asked about their code:**  
1.  
2.  
3.  

### b) Team B's review of our code (step 4b)  
**Questions they asked about our code and how we answered:**  
1.  
2.  
3.  

### c) Notes and insights on Team B's talking points (we take notes when they present)  
-  
-  
-  

### d) Notes and insights Team B wrote on our talking points (when we present)  
-  
-  
-  

## 11. Report Details (for submission)  

| Item | Value |
|---|---|
| Workshop title | Statistical Classification Log-Loss (Cross-Entropy): Logistic Regression Classifier Workshop |
| Team A members | Sultan Akinola Atanda (9114837), _Member 2_, _Member 3_, _Member 4_ |
| Team A repo | _add link_ |
| Team B members | _add names_ |
| Team B repo | _add link_ |

## Summary  

- **Sigmoid** turns a straight-line score into a probability between 0 and 1.  
- **Logistic regression** learns the line's weights; a **0.5 threshold** turns probabilities into Pass/Fail.  
- **Log-Loss** scores the probabilities: low when confident and right, very high when confident and wrong.  
- We built Log-Loss **from scratch**, confirmed it matches scikit-learn, and showed it is the quantity **gradient descent minimizes** to train the model.  